# DRAEM — Spacespresso

Discriminatively-trained Reconstruction Anomaly Embedding Model.
Semi-supervised: labeled GT masks fine-tune the discriminator.

See `src/methods/draem.py` for the implementation.

## Setup

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd().resolve()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from src.common.config import load_config
from src.common.data import SpacepressoDataModule
from src.common.evaluation import evaluate_predictions
from src.common.paths import resolve_path
from src.common.seed import set_seed
from src.common.visualization import show_predictions


In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device:", torch.cuda.get_device_name(0))
print("PyTorch:", torch.__version__)


## Configuration

In [ ]:
from src.methods.draem import Method as DraemMethod

draem_cfg = load_config(ROOT / "configs/draem/jasmin_draem.yaml")

local_data = ROOT / "data" / "spacepresso"
if local_data.exists():
    draem_cfg["data"]["root"] = str(local_data)
draem_cfg["data"]["load_images"] = False

# --- Toggles ---
NO_BACKGROUND          = True
MULTI_VIEW_FUSION      = True
MULTI_VIEW_CONSISTENCY = True
CONSISTENCY_ALPHA      = 0.5
BG_DILATION            = 16
BG_THRESHOLD           = 0.20
BG_THRESHOLD_PER_CLASS = {
    "class_01": 0.40,
    "class_02": 0.30,
    "class_08": 0.40,
}
VAL_ANOMALY_FRACTION = 0.2   # hold out 20% of anomalies for local AP check
SAVE_FILE            = True

def get_bg_threshold(class_name):
    return BG_THRESHOLD_PER_CLASS.get(class_name, BG_THRESHOLD)


## Data

In [ ]:
set_seed(draem_cfg.get("seed", 42))

dm = SpacepressoDataModule(**draem_cfg["data"])
train_good      = dm.load_train_good()
train_anomalies = dm.load_train_anomalies()
test            = dm.load_test()
print({"train_good": len(train_good), "train_anomalies": len(train_anomalies), "test": len(test)})

import random as _rng
_rng.seed(draem_cfg.get("seed", 42))
_shuffled = _rng.sample(train_anomalies, len(train_anomalies))
_n_val = max(1, int(len(_shuffled) * VAL_ANOMALY_FRACTION)) if VAL_ANOMALY_FRACTION > 0 else 0
train_anomalies_val = _shuffled[:_n_val]
train_anomalies_fit = _shuffled[_n_val:]
print(f"Anomaly split: {len(train_anomalies_fit)} fit / {len(train_anomalies_val)} val")


## Training

In [ ]:
# --- Train DRAEM ---
draem_runner = DraemMethod(draem_cfg)
draem_runner.fit(train_good + train_anomalies_fit)
draem_predictions = draem_runner.predict(test)
show_predictions(test, draem_predictions, n=3)


## Post-processing

In [ ]:
import numpy as np
from collections import defaultdict
from scipy.ndimage import binary_fill_holes, binary_dilation as _bdilate
from PIL import Image as PILImage

predictions = draem_predictions
image_size  = draem_cfg["data"]["image_size"]

# Background suppression
if NO_BACKGROUND and predictions:
    id_to_sample = {s.image_id: s for s in test}
    for img_id, pred in predictions.items():
        s   = id_to_sample[img_id]
        thr = get_bg_threshold(s.class_name)
        img = np.array(PILImage.open(s.image_path).convert("RGB").resize((image_size, image_size))) / 255.0
        fg  = binary_fill_holes(img.mean(axis=2) > thr)
        if BG_DILATION > 0: fg = _bdilate(fg, iterations=BG_DILATION)
        predictions[img_id] = pred * fg.astype(np.float32)
    print("Background suppression applied.")

# Multi-view fusion
sample_to_views = defaultdict(list)
for s in test: sample_to_views[s.sample_id].append(s.image_id)

if MULTI_VIEW_FUSION and predictions:
    n_boosted = 0
    for image_ids in sample_to_views.values():
        maps = [predictions[iid] for iid in image_ids if iid in predictions]
        if not maps: continue
        sample_max = max(float(m.max()) for m in maps)
        for iid in image_ids:
            if iid not in predictions: continue
            view_max = float(predictions[iid].max())
            if view_max < sample_max - 1e-6:
                predictions[iid] = np.clip(predictions[iid] * (sample_max / max(view_max, 1e-6)), 0.0, 1.0).astype(predictions[iid].dtype)
                n_boosted += 1
    print(f"Multi-view fusion: {n_boosted} views boosted.")

if MULTI_VIEW_CONSISTENCY and predictions:
    n_boosted = 0
    for image_ids in sample_to_views.values():
        present = [iid for iid in image_ids if iid in predictions]
        if len(present) < 2: continue
        scores = [float(predictions[iid].max()) for iid in present]
        s_max, s_mean = max(scores), float(np.mean(scores))
        if s_mean < 1e-6: continue
        boost = 1.0 + CONSISTENCY_ALPHA * (s_max - s_mean) / (s_max + 1e-6)
        for iid in present:
            predictions[iid] = np.clip(predictions[iid] * boost, 0.0, 1.0).astype(predictions[iid].dtype)
            n_boosted += 1
    print(f"Multi-view consistency: {n_boosted} views boosted.")


## Validation

In [ ]:
if train_anomalies_val:
    import random as _random
    import matplotlib.pyplot as plt
    from scipy.ndimage import binary_fill_holes, binary_dilation as _bdilate
    from src.common.evaluation import load_sample_mask

    N_GOOD_VAL  = 200
    good_sample = _random.sample(train_good, min(N_GOOD_VAL, len(train_good)))
    val_samples = train_anomalies_val + good_sample
    val_pred    = draem_runner.predict(val_samples)

    # Mirror post-processing
    if NO_BACKGROUND:
        _id2s = {s.image_id: s for s in val_samples}
        for img_id, pred in val_pred.items():
            s = _id2s[img_id]; thr = get_bg_threshold(s.class_name)
            img = np.array(PILImage.open(s.image_path).convert("RGB").resize((image_size, image_size))) / 255.0
            fg = binary_fill_holes(img.mean(axis=2) > thr)
            if BG_DILATION > 0: fg = _bdilate(fg, iterations=BG_DILATION)
            val_pred[img_id] = pred * fg.astype(np.float32)

    val_s2v = defaultdict(list)
    for s in val_samples: val_s2v[s.sample_id].append(s.image_id)
    if MULTI_VIEW_FUSION:
        for iids in val_s2v.values():
            maps = [val_pred[i] for i in iids if i in val_pred]
            if not maps: continue
            sm = max(float(m.max()) for m in maps)
            for i in iids:
                if i not in val_pred: continue
                vm = float(val_pred[i].max())
                if vm < sm - 1e-6:
                    val_pred[i] = np.clip(val_pred[i] * (sm / max(vm, 1e-6)), 0.0, 1.0).astype(val_pred[i].dtype)
    if MULTI_VIEW_CONSISTENCY:
        for iids in val_s2v.values():
            present = [i for i in iids if i in val_pred]
            if len(present) < 2: continue
            scores = [float(val_pred[i].max()) for i in present]
            sm, smean = max(scores), float(np.mean(scores))
            if smean < 1e-6: continue
            boost = 1.0 + CONSISTENCY_ALPHA * (sm - smean) / (sm + 1e-6)
            for i in present:
                val_pred[i] = np.clip(val_pred[i] * boost, 0.0, 1.0).astype(val_pred[i].dtype)

    metrics = evaluate_predictions(val_samples, val_pred).as_dict()
    print(f"[DRAEM] {len(train_anomalies_val)} anomalies + {len(good_sample)} good")
    print(f"Pixel AP:    {metrics['pixel_ap']:.4f}")
    print(f"Image AP:    {metrics['image_ap']:.4f}")
    print(f"Pixel AUROC: {metrics['pixel_auroc']:.4f}")

    # Visualise anomalies
    grouped = {}
    for s in train_anomalies_val: grouped.setdefault(s.class_name, []).append(s)
    if grouped:
        N_ANOM = 4
        n_cls = len(grouped)
        fig, axes = plt.subplots(n_cls, N_ANOM * 3, figsize=(N_ANOM * 9, 3 * n_cls), squeeze=False)
        for row, (cls, slist) in enumerate(sorted(grouped.items())):
            picks = _random.sample(slist, min(N_ANOM, len(slist)))
            for col_off, s in enumerate(picks):
                img  = np.array(PILImage.open(s.image_path).convert("RGB").resize((image_size, image_size))) / 255.0
                gt   = load_sample_mask(s, (image_size, image_size)).astype(np.float32)
                pred = np.asarray(val_pred[s.image_id], dtype=np.float32)
                c = col_off * 3
                axes[row,c].imshow(img); axes[row,c].axis("off")
                axes[row,c].set_title(cls if col_off==0 else "", fontsize=7)
                axes[row,c+1].imshow(gt, cmap="hot", vmin=0, vmax=1); axes[row,c+1].axis("off")
                axes[row,c+1].set_title("GT" if row==0 else "", fontsize=7)
                axes[row,c+2].imshow(pred, cmap="hot", vmin=0, vmax=1); axes[row,c+2].axis("off")
                axes[row,c+2].set_title("pred" if row==0 else "", fontsize=7)
        plt.suptitle("DRAEM — anomalies", fontsize=9); plt.tight_layout(); plt.show()

    # Visualise normals
    N_NORM = 10
    norm_picks = _random.sample(good_sample, min(N_NORM, len(good_sample)))
    fig, axes = plt.subplots(2, N_NORM, figsize=(2*N_NORM, 6))
    for col, s in enumerate(norm_picks):
        img  = np.array(PILImage.open(s.image_path).convert("RGB").resize((image_size, image_size))) / 255.0
        pred = np.asarray(val_pred[s.image_id], dtype=np.float32)
        axes[0,col].imshow(img); axes[0,col].axis("off"); axes[0,col].set_title(s.class_name, fontsize=6)
        axes[1,col].imshow(pred, cmap="hot", vmin=0, vmax=1); axes[1,col].axis("off")
        axes[1,col].set_title(f"max={pred.max():.3f}", fontsize=6)
    plt.suptitle("Normal images — should score near zero", fontsize=9); plt.tight_layout(); plt.show()
else:
    print("No held-out anomalies — set VAL_ANOMALY_FRACTION > 0 for local AP")


## Save submission

In [ ]:
if predictions and SAVE_FILE:
    import zipfile, random
    from datetime import datetime
    from src.common.submission import _prepare_prediction_map
    from src.common.q8rle import float_matrix_to_q8rle, q8rle_to_float_matrix

    expected_shape = (image_size, image_size)
    sorted_ids = sorted(predictions)
    assert len(sorted_ids) == len(test)

    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    base_path = resolve_path(draem_cfg["submission"]["output_path"], ROOT)
    zip_path  = base_path.parent / f"draem_{timestamp}.zip"
    zip_path.parent.mkdir(parents=True, exist_ok=True)

    spot = set(random.sample(range(len(sorted_ids)), min(5, len(sorted_ids))))
    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED, compresslevel=6) as zf:
        with zf.open(f"draem_{timestamp}.csv", "w") as f:
            f.write(b"ID,Label\n")
            for i, img_id in enumerate(sorted_ids):
                label = float_matrix_to_q8rle(_prepare_prediction_map(predictions[img_id]))
                f.write(f"{img_id},{label}\n".encode("utf-8"))
                if i in spot:
                    assert q8rle_to_float_matrix(label).shape == expected_shape

    print(f"Saved: {zip_path} ({zip_path.stat().st_size / 1024**2:.1f} MB)")
